In [33]:
from dotenv import load_dotenv
load_dotenv()

True

In [34]:
#!pip install langchain langchain-community faiss-cpu PyPDF2 sentence-transformers

In [35]:
#!pip install ollama

In [36]:
import os
import glob
import re
from langchain_community.document_loaders import PyPDFLoader, TextLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter
import ollama
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from langchain.chains import RetrievalQA, ConversationalRetrievalChain
from langchain_community.llms import Ollama
from typing import List, Optional, Tuple
from langchain.prompts import PromptTemplate

pdf_directory = "./pdf"
txt_directory = "./txt"

documents = []

pdf_files = glob.glob(os.path.join(pdf_directory, "*.pdf"))
print(f"{len(pdf_files)}개의 PDF 파일을 발견했습니다.")

for pdf_file in pdf_files:
    loader = PyPDFLoader(pdf_file)
    documents.extend(loader.load())

txt_files = glob.glob(os.path.join(txt_directory, "*.txt"))
print(f"{len(txt_files)}개의 TXT 파일을 발견했습니다.")

for txt_file in txt_files:
    loader = TextLoader(txt_file, encoding='utf-8')  # 한글 인코딩 지원
    documents.extend(loader.load())
print(f"총 {len(documents)}개의 문서 로드 완료")

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200,
    length_function=len,
)

chunks = text_splitter.split_documents(documents)
print(f"총 {len(chunks)}개의 청크로 분할 완료")

46개의 PDF 파일을 발견했습니다.
6개의 TXT 파일을 발견했습니다.
총 368개의 문서 로드 완료
총 596개의 청크로 분할 완료


In [37]:
embeddings = HuggingFaceEmbeddings(
    model_name="jhgan/ko-sroberta-multitask",
    model_kwargs={'device': 'cpu'},     # GPU 있으면 'cuda'로 변경
    encode_kwargs={'normalize_embeddings': True}
)

In [38]:
# FAISS 벡터 저장소 생성
vector_store = FAISS.from_documents(chunks, embeddings)

# 벡터 저장소 저장 (나중에 재사용할 수 있도록)
vector_store.save_local("faiss_index")
print("벡터 저장소가 'faiss_index' 폴더에 저장되었습니다.")

# Ollama 모델 설정 (gemma3:1b 사용)
llm = Ollama(model="gemma3:1b")

벡터 저장소가 'faiss_index' 폴더에 저장되었습니다.


In [ ]:
# 개선된 시스템 프롬프트 - 보다 명확한 지시와 형식을 제공
SYSTEM_PROMPT = """
당신은 여행 금지 품목 및 제한사항에 대한 전문적인 지식을 갖춘 챗봇입니다.
각국의 입국 시 금지/제한되는 품목에 대해 정확한 정보를 제공해야 합니다.

중요 지침:
1. 사용자 질문을 정확히 분석하고 관련 정보만 제공하세요.
2. 질문에 언급된 국가와 품목에 대해서만 답변하세요.
3. 검색된 문서에 관련 정보가 없으면 "해당 정보를 찾을 수 없습니다. 공식 웹사이트나 대사관에 문의하시는 것이 좋겠습니다."라고 답변하세요.
4. 사용자가 국가를 명시하지 않으면 "어떤 국가에 대한 정보가 필요하신가요?"라고 명확히 질문하세요.
5. 답변은 3-5문장으로 간결하게 제공하세요.
6. 질문과 관련 없는 정보는 포함하지 마세요.
7. 불확실한 정보는 제공하지 마세요.

항공기 탑승 시 일반적인 제한사항:
- 액체류: 개별 용기 100ml 이하, 1L 투명 지퍼백에 담아야 함
- 예외: 유아식, 의약품 등(보안 검색 시 별도 신고)
- 위험물: 라이터(1개만 기내 반입 가능), 성냥, 스프레이, 압축 가스
- 날카로운 물건: 칼, 가위, 면도칼 등은 위탁 수하물로만 가능
"""

# 프롬프트 템플릿 개선 - 더 명확한 형식과 지시 제공
PROMPT_TEMPLATE = """
{system_prompt}

사용자 질문: {question}

다음은 질문과 관련된 문서 정보입니다:
{context}

1. 위 정보를 바탕으로 사용자 질문에 직접적으로 관련된 내용만 답변하세요.
2. 답변은 3-5문장으로 간결하게 작성하세요.
3. 질문과 관련 없는 정보는 포함하지 마세요.
4. 불확실한 정보는 제공하지 말고 "해당 정보를 찾을 수 없습니다"라고 명시하세요.

답변:
"""

# 국가와 물품 감지 함수 개선 - 더 포괄적이고 정확한 감지
class EntityDetector:
    def __init__(self):
        # 국가 목록 확장 - 영어 이름과 다양한 표현 추가
        self.countries = {
            "일본": ["일본", "japan", "japanese", "jp", "니혼", "닛폰"],
            "미국": ["미국", "usa", "united states", "america", "american", "us", "유에스에이", "아메리카"],
            "중국": ["중국", "china", "chinese", "cn", "차이나"],
            "태국": ["태국", "thailand", "thai", "th"],
            "호주": ["호주", "australia", "australian", "au", "오스트레일리아"],
            "한국": ["한국", "korea", "korean", "kr", "대한민국", "코리아"],
            "영국": ["영국", "united kingdom", "uk", "britain", "british", "gb", "잉글랜드", "브리튼"],
            "프랑스": ["프랑스", "france", "french", "fr"],
            "독일": ["독일", "germany", "german", "de"],
            "이탈리아": ["이탈리아", "italy", "italian", "it"],
            "스페인": ["스페인", "spain", "spanish", "es"],
            "캐나다": ["캐나다", "canada", "canadian", "ca"],
            "싱가포르": ["싱가포르", "singapore", "singaporean", "sg"],
            "말레이시아": ["말레이시아", "malaysia", "malaysian", "my"],
            "인도네시아": ["인도네시아", "indonesia", "indonesian", "id"],
            "베트남": ["베트남", "vietnam", "vietnamese", "vn"],
            "필리핀": ["필리핀", "philippines", "philippine", "filipino", "ph"]
        }
        
        # 품목 목록 확장 - 다양한 표현과 범주 포함
        self.items = {
            "액체류": ["액체", "물", "음료", "주스", "음료수", "화장품", "샴푸", "로션", "향수"],
            "식품류": ["음식", "식품", "과일", "채소", "육류", "생선", "해산물", "유제품", "간식", "과자"],
            "의약품": ["약", "의약품", "처방약", "알약", "진통제", "비타민", "영양제", "보조제"],
            "주류": ["술", "주류", "알코올", "맥주", "와인", "소주", "위스키", "보드카"],
            "담배류": ["담배", "시가", "전자담배", "베이프", "흡연", "니코틴"],
            "전자제품": ["전자제품", "노트북", "컴퓨터", "휴대폰", "카메라", "배터리", "충전기", "전자기기"],
            "무기류": ["칼", "가위", "면도기", "날카로운", "무기", "총", "총기", "폭발물"],
            "식물류": ["식물", "씨앗", "화초", "꽃", "나무", "농산물"],
            "동물류": ["동물", "애완동물", "반려동물", "강아지", "고양이", "새", "어류"],
            "위조품": ["가짜", "위조", "복제품", "짝퉁", "모조품"],
            "수하물": ["수하물", "짐", "캐리어", "가방", "여행가방"]
        }
    
    def detect_country(self, text: str) -> List[str]:
        """텍스트에서 국가명 감지 - 더 정확한 매칭을 위해 개선"""
        detected = []
        
        # 소문자로 변환하여 대소문자 구분 문제 해결
        text_lower = text.lower()
        
        for country, aliases in self.countries.items():
            for alias in aliases:
                # 단어 경계를 고려한 매칭 (부분 문자열이 아닌 온전한 단어로 매칭)
                if re.search(r'\b' + re.escape(alias.lower()) + r'\b', text_lower):
                    detected.append(country)
                    break
        
        return detected
    
    def detect_item(self, text: str) -> List[str]:
        """텍스트에서 품목명 감지 - 더 정확한 매칭을 위해 개선"""
        detected = []
        
        # 소문자로 변환하여 대소문자 구분 문제 해결
        text_lower = text.lower()
        
        for item, aliases in self.items.items():
            for alias in aliases:
                # 단어 경계를 고려한 매칭 (부분 문자열이 아닌 온전한 단어로 매칭)
                if re.search(r'\b' + re.escape(alias.lower()) + r'\b', text_lower):
                    detected.append(item)
                    break
        
        return detected

# 문맥 관련성 평가 함수 추가
def evaluate_relevance(query: str, doc_content: str) -> float:
    """
    쿼리와 문서 콘텐츠 간의 관련성 점수를 계산
    
    Args:
        query: 사용자 질문
        doc_content: 검색된 문서 내용
        
    Returns:
        관련성 점수 (0.0~1.0)
    """
    # 간단한 키워드 기반 점수 계산 (실제로는 더 정교한 알고리즘 필요)
    query_words = set(re.findall(r'\b\w+\b', query.lower()))
    doc_words = set(re.findall(r'\b\w+\b', doc_content.lower()))
    
    # 공통 단어 수
    common_words = query_words.intersection(doc_words)
    
    # 단순 자카드 유사도
    if not query_words:
        return 0.0
    
    return len(common_words) / len(query_words)

# 쿼리 확장 및 필터링 클래스
class QueryProcessor:
    def __init__(self, entity_detector: EntityDetector):
        self.entity_detector = entity_detector
    
    def enhance_query(self, query: str) -> str:
        """
        사용자 질문을 확장하여 더 관련성 높은 문서를 검색하도록 함
        """
        countries = self.entity_detector.detect_country(query)
        items = self.entity_detector.detect_item(query)
        
        # 기본 키워드 추가
        enhanced_query = query
        
        # 항공 또는 여행 관련 키워드가 없으면 추가
        if not any(keyword in query.lower() for keyword in ["항공", "비행기", "여행", "입국", "세관", "반입"]):
            enhanced_query += " 항공 여행 입국"
        
        # 금지 또는 제한 관련 키워드가 없으면 추가
        if not any(keyword in query.lower() for keyword in ["금지", "제한", "반입금지", "통관", "규제"]):
            enhanced_query += " 금지 제한 품목"
        
        # 국가가 감지되면 국가 관련 키워드 추가
        if countries:
            country_terms = " ".join(countries)
            enhanced_query += f" {country_terms} 입국"
        
        # 품목이 감지되면 품목 관련 키워드 추가
        if items:
            item_terms = " ".join(items)
            enhanced_query += f" {item_terms} 반입"
        
        return enhanced_query
    
    def filter_documents(self, query: str, docs: List[str], threshold: float = 0.2) -> List[str]:
        """
        검색된 문서 중 관련성이 높은 문서만 필터링
        
        Args:
            query: 원본 사용자 질문
            docs: 검색된 문서 목록
            threshold: 관련성 임계값 (이 값 이상만 유지)
            
        Returns:
            필터링된 문서 목록
        """
        filtered_docs = []
        
        for doc in docs:
            relevance = evaluate_relevance(query, doc)
            if relevance >= threshold:
                filtered_docs.append(doc)
        
        # 모든 문서가 필터링되었다면 가장 관련성 높은 문서 하나라도 포함
        if not filtered_docs and docs:
            # 각 문서의 관련성 점수 계산
            relevance_scores = [(doc, evaluate_relevance(query, doc)) for doc in docs]
            # 관련성 점수 기준으로 정렬
            sorted_docs = sorted(relevance_scores, key=lambda x: x[1], reverse=True)
            # 가장 점수가 높은 문서 추가
            filtered_docs.append(sorted_docs[0][0])
        
        return filtered_docs

# 챗봇 응답 검증 및 포맷팅 클래스
class ResponseFormatter:
    def __init__(self):
        # 과도하게 긴 응답 감지를 위한 임계값
        self.max_paragraphs = 2
        self.max_sentences = 8
    
    def validate_response(self, response: str) -> bool:
        """
        응답이 적절한지 검증
        
        Args:
            response: 생성된 응답
            
        Returns:
            응답이 적절하면 True, 그렇지 않으면 False
        """
        # 응답이 비어있거나 너무 짧은 경우
        if not response or len(response) < 10:
            return False
        
        # 과도하게 많은 단락이 있는 경우
        paragraphs = response.split('\n\n')
        if len(paragraphs) > self.max_paragraphs:
            return False
        
        # 과도하게 많은 문장이 있는 경우
        sentences = re.split(r'[.!?]+', response)
        if len(sentences) > self.max_sentences:
            return False
        
        return True
    
    def format_response(self, response: str) -> str:
        """
        응답을 더 나은 형식으로 정리
        
        Args:
            response: 원본 응답
            
        Returns:
            포맷팅된 응답
        """
        # 중복된 공백 제거
        formatted = re.sub(r'\s+', ' ', response).strip()
        
        # 불필요한 마크다운 또는 특수 문자 제거
        formatted = re.sub(r'#+\s+', '', formatted)  # 마크다운 헤더 제거
        formatted = re.sub(r'\*+', '', formatted)    # 마크다운 볼드/이탤릭 제거
        
        # 출처 또는 참고문헌 관련 문구 제거
        formatted = re.sub(r'출처:.*', '', formatted)
        formatted = re.sub(r'참고:.*', '', formatted)
        formatted = re.sub(r'참고문헌:.*', '', formatted)
        
        # 링크 형식 제거
        formatted = re.sub(r'\[.*?\]\(.*?\)', '', formatted)
        
        # 과도한 문장 제거 (최대 5개 문장으로 제한)
        sentences = re.split(r'([.!?]+)', formatted)
        if len(sentences) > 10:  # 구분자 포함하여 계산하므로 2배로 체크
            formatted_sentences = []
            for i in range(min(10, len(sentences) - 1)):
                formatted_sentences.append(sentences[i])
            formatted = ''.join(formatted_sentences)
        
        return formatted.strip()

# 개선된 여행 금지 품목 챗봇 클래스
class ImprovedTravelRestrictionsBot:
    def __init__(self, vector_store):
        self.vector_store = vector_store
        self.retriever = vector_store.as_retriever(search_kwargs={"k": 5})  # 더 많은 문서 검색
        self.entity_detector = EntityDetector()
        self.query_processor = QueryProcessor(self.entity_detector)
        self.response_formatter = ResponseFormatter()
        
        # LangChain 프롬프트 템플릿 설정
        self.prompt = PromptTemplate(
            input_variables=["system_prompt", "question", "context"],
            template=PROMPT_TEMPLATE
        )
    
    def check_input_validity(self, query: str) -> Tuple[bool, Optional[str]]:
        """
        입력이 유효한지 확인하고 필요시 즉시 응답 생성
        
        Args:
            query: 사용자 질문
            
        Returns:
            (유효성 여부, 즉시 응답(있는 경우))
        """
        # 빈 질문 처리
        if not query.strip():
            return False, "질문을 입력해주세요."
        
        # 너무 짧은 질문 처리
        if len(query.strip()) < 3:
            return False, "좀 더 자세한 질문을 입력해주세요."
        
        # 국가 명시 여부 확인
        countries = self.entity_detector.detect_country(query)
        
        # "낙타" 관련 특수 질문 처리 (이전 예시에서 문제가 된 질문)
        if "낙타" in query and "타" in query:
            return False, "여행 시 특정 동물을 타는 것에 관한 정보는 제공하지 않습니다. 국가별 반입 금지 품목이나 여행 규정에 대해 질문해주세요."
        
        # 국가가 없지만 일반적인 항공 규정에 관한 질문인 경우
        if not countries and any(keyword in query.lower() for keyword in ["항공", "비행기", "기내", "위탁", "수하물", "반입"]):
            return True, None  # 유효함, 일반 항공 규정으로 처리할 것
            
        # 국가가 없는 경우 국가 지정 요청
        if not countries and not "일반" in query:
            return False, "어떤 국가에 대한 정보가 필요하신가요? 국가명을 포함하여 질문해주세요."
        
        return True, None  # 유효함, 정상 처리할 것
    
    def get_response(self, query: str) -> str:
        """
        사용자 질문에 대한 응답 생성 (개선된 버전)
        
        Args:
            query: 사용자 질문
            
        Returns:
            생성된 응답
        """
        # 입력 유효성 검사
        is_valid, immediate_response = self.check_input_validity(query)
        if not is_valid:
            return immediate_response
        
        try:
            # 질문 강화
            enhanced_query = self.query_processor.enhance_query(query)
            
            # 관련 문서 검색
            docs = self.retriever.get_relevant_documents(enhanced_query)
            doc_contents = [doc.page_content for doc in docs]
            
            # 관련성 기반 문서 필터링
            filtered_docs = self.query_processor.filter_documents(query, doc_contents)
            
            # 검색된 문서가 없는 경우
            if not filtered_docs:
                return "죄송합니다. 해당 질문에 대한 정보를 찾을 수 없습니다. 다른 방식으로 질문해주시거나 공식 여행 정보 사이트를 확인해보세요."
            
            # 문서 컨텍스트 준비
            context = "\n\n---\n\n".join(filtered_docs)
            
            # Ollama API 호출을 위한 메시지 준비
            messages = [
                {
                    "role": "system",
                    "content": SYSTEM_PROMPT
                },
                {
                    "role": "user",
                    "content": f"질문: {query}\n\n다음은 참고할 정보입니다:\n{context}\n\n3-5문장으로 간결하게 답변해주세요."
                }
            ]
            
            # Ollama API 호출 - 더 큰 모델 사용 (가능하다면)
            # 참고: 실제 환경에서는 사용 가능한 더 큰 모델로 교체하는 것이 좋습니다
            # model_to_use = "gemma3:4b" if "gemma3:4b" in ollama.list()['models'] else "gemma3:1b"
            model_to_use = "gemma3:1b"
            response = ollama.chat(
                model=model_to_use,
                messages=messages
            )
            
            raw_response = response["message"]["content"]
            
            # 응답 검증 및 포맷팅
            if not self.response_formatter.validate_response(raw_response):
                # 응답이 적절하지 않은 경우 간단한 응답으로 대체
                return "죄송합니다. 해당 질문에 대한 정확한 정보를 제공할 수 없습니다. 더 구체적인 질문을 해주시거나 공식 정보를 확인해보세요."
            
            formatted_response = self.response_formatter.format_response(raw_response)
            return formatted_response
            
        except Exception as e:
            # 오류 발생 시 안전한 응답 제공
            print(f"오류 발생: {str(e)}")
            return "죄송합니다. 응답 생성 중 오류가 발생했습니다. 다시 시도해주세요."

# 챗봇 인스턴스 생성 및 사용 예시
def setup_chatbot(vector_store_path="faiss_index"):
    """
    벡터 저장소에서 챗봇 인스턴스 생성
    
    Args:
        vector_store_path: 벡터 저장소 경로
        
    Returns:
        챗봇 인스턴스
    """
    # 임베딩 모델 로드
    embeddings = HuggingFaceEmbeddings(
        model_name="jhgan/ko-sroberta-multitask",
        model_kwargs={'device': 'cpu'},
        encode_kwargs={'normalize_embeddings': True}
    )
    
    # 벡터 저장소 로드
    vector_store = FAISS.load_local(
        vector_store_path, 
        embeddings,
        allow_dangerous_deserialization=True
    )
    
    # 챗봇 인스턴스 생성
    return ImprovedTravelRestrictionsBot(vector_store)

# 대화형 모드 개선 - 더 안정적이고 사용자 친화적인 인터페이스
def interactive_chat_mode(bot):
    """
    대화형 모드로 챗봇 실행
    
    Args:
        bot: 챗봇 인스턴스
    """
    print("=== 개선된 여행 금지 품목 확인 챗봇 ===")
    print("종료하려면 'q' 또는 'exit' 또는 '종료'를 입력하세요.")
    print("="*50)
    print("\n안녕하세요! 여행 시 금지/제한 품목에 대해 궁금한 점을 질문해주세요.")
    print("예시 질문: '일본에 과일을 가져갈 수 있나요?', '미국 입국 시 가져가면 안되는 물건은?'\n")
    
    while True:
        try:
            user_input = input("\n질문을 입력하세요: ")
            print(f'\n질문: {user_input}')
            user_input = user_input.strip()
            
            if user_input.lower() in ['q', 'exit', '종료', 'quit']:
                print("챗봇을 종료합니다. 좋은 여행 되세요!")
                break
                
            print("\n답변: ", end="")
            response = bot.get_response(user_input)
            print(f"{response}")
            
        except KeyboardInterrupt:
            print("\n\n사용자에 의해 챗봇이 종료되었습니다.")
            break
        except Exception as e:
            print(f"\n오류가 발생했습니다: {str(e)}")
            print("다시 시도해주세요.")

# 메인 실행 함수
def main():
    # 벡터 저장소가 이미 생성되어 있다면 바로 로드
    if os.path.exists("faiss_index"):
        bot = setup_chatbot()
        interactive_chat_mode(bot)
    else:
        print("벡터 저장소를 찾을 수 없습니다. 먼저 데이터를 로드하고 벡터 저장소를 생성해주세요.")

if __name__ == "__main__":
    main()

=== 개선된 여행 금지 품목 확인 챗봇 ===
종료하려면 'q' 또는 'exit' 또는 '종료'를 입력하세요.

안녕하세요! 여행 시 금지/제한 품목에 대해 궁금한 점을 질문해주세요.
예시 질문: '일본에 과일을 가져갈 수 있나요?', '미국 입국 시 가져가면 안되는 물건은?'


질문: 미국 놀러갈건데 뭐해야하지?

답변: 미국 방문 시, 액체류, 총기, 고기, 농산물 등 일부 식품은 반입이 제한됩니다. 미국 입국 시 면세 및 반입 관련 규정을 확인하고, 특히 선물은 사전 신고가 필요하지 않습니다. 자세한 내용은 미국 세관 홈페이지를 참고하시기 바랍니다.
